# Modified VIEScore for RefEdit-Bench

Scores **existing generated RefEdit outputs**; this notebook does not run the editing agent. Select a run with `MMEDIT_REFEDIT_RUN` or leave it empty to score the latest run. It reads that run's `results/results.jsonl` and `results/images/*_output.png`, so saved images remain discoverable if a subset run replaced the JSONL summary. Scoring uses RefEdit's `metadata.edit_instruction_single`, the dataset GT polygon mask, and identical zero-padding tight source/output crops. Scores are stored separately in that run's `metrics/modified_viescore.jsonl`; inference never creates this directory.

Install GEdit-Bench VIEScore dependencies in this notebook's Python environment (including `regex` and `requests`), clone [Step1X-Edit](https://github.com/stepfun-ai/Step1X-Edit), and set `VIESCORE_ROOT` to its `GEdit-Bench` directory. Set `VIESCORE_KEY_PATH` to a local file containing only your OpenAI API key on the first line. The scorer requests the `gpt-4o` API model; API usage may incur charges. Never put the key in a cell or commit its file.

In [ ]:
import os
from pathlib import Path

from evaluation.notebook import load_benchmark, resolve_run_directory

BENCHMARK = "refedit_bench"
context = load_benchmark(BENCHMARK)
ANNOTATION_FILE = context.annotation_file
OUTPUT_ROOT = context.output_dir
RUN_NAME = os.environ.get("MMEDIT_REFEDIT_BENCH_RUN") or None
OUTPUT_DIR = resolve_run_directory(OUTPUT_ROOT, RUN_NAME)
records = context.records
image_loader = context.image_loader
if not OUTPUT_DIR.is_dir():
    raise FileNotFoundError(f"RefEdit output directory not found: {OUTPUT_DIR}")
VIESCORE_ROOT = Path(os.environ.get("VIESCORE_ROOT") or ".").expanduser()
VIESCORE_KEY_PATH = Path(os.environ.get("VIESCORE_KEY_PATH") or ".").expanduser()
print("Annotations:", ANNOTATION_FILE)
print("Selected run:", OUTPUT_DIR)
print("VIEScore checkout:", VIESCORE_ROOT if str(VIESCORE_ROOT) != "." else "not configured")
print("API key file:", VIESCORE_KEY_PATH if str(VIESCORE_KEY_PATH) != "." else "not configured")

In [ ]:
from evaluation.common import HFImageLoader, read_jsonl
from evaluation.refedit_bench_viescore import collect_saved_outputs

records = list(read_jsonl(ANNOTATION_FILE))
if not records:
    raise ValueError(f'No RefEdit annotations found in {ANNOTATION_FILE}')
image_loader = HFImageLoader()
saved_outputs = collect_saved_outputs(
    records,
    OUTPUT_DIR / 'results' / 'results.jsonl',
    OUTPUT_DIR / 'results' / 'images',
)
print(f'Found {len(saved_outputs)} saved output images:')
for saved in saved_outputs:
    print(f"  [{saved['annotation_index']}] {saved['sample_id']} ({saved['discovered_from']})")

## Score saved outputs

Run this cell after reviewing the discovered sample list above. Unchanged images reuse cached VIEScore rows; changed or newly discovered images are evaluated and appended to the output JSONL.

## Configure VIEScore

This uses the upstream VIEScore `tie` task with the GPT-4o API model. The API key is read from the local key file; it is never displayed.

## Score saved outputs

The loop below checks annotation/dataset consistency, reuses unchanged cached rows, scores uncached outputs, and atomically rewrites the JSONL after each sample so progress is preserved.

In [ ]:
from IPython.display import display
from evaluation.refedit_bench_viescore import evaluate_saved_refedit_bench_outputs

metric_rows, score_summary = evaluate_saved_refedit_bench_outputs(
    records,
    output_dir=OUTPUT_DIR,
    image_loader=image_loader,
    viescore_root=VIESCORE_ROOT,
    key_path=VIESCORE_KEY_PATH,
)
print(f"Per-sample Modified VIEScore: {OUTPUT_DIR / 'metrics' / 'modified_viescore.jsonl'}")
display([{"category": category, **metrics} for category, metrics in score_summary.items()])